# TavilySearch 사용
- VectorDB없이 웹 겁색 결과를 문맥으로 넣어 LLM이 답하도록 하는 웹 검색기반 RAG구조에 적합
- 실시간 웹 검색 구현
- 트래픽 급증 속 'No-AI' 검색 엔진에 더  쉽게 접근할 수 있게 함
- https://www.tavily.com

In [2]:
!pip install -U langchain-openai langchain-community
!pip install -U langchain-tavily # Tavily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [3]:
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.prompts import ChatPromptTemplate
import time
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
class MyOptimuzerWebRag:
  def __init__(self):
    # 검색 도구 생성
    self.search = TavilySearch(
        max_resultes = 5,
        api_key = os.getenv("TAVILY_API_KEY"),

    )
    self.llm = ChatOpenAI(
        model = 'gpt-4o-mini',
        temperature=0.0, # 일관된 답변을 생성(정형적)
        api_key=os.getenv("OPENAI_API_KEY")
    )

    self.summary_prompt = ChatPromptTemplate.from_template(
        """
          당신은 검색 결과를 정리 요약하는 전문가 입니다.

          다음은 웹 검색 결과입니다.
          {search_results}

          [요구사항]
            - 광고/홍보성 내용을 제거
            - 서로 겹치는 내용은 합치기
            - 핵심 정보만 5줄 내외로 간결하게 정리하세요

          [검색 결과 요약]

        """
    )
    # 최종 답변용 prompt
    self.answer_prompt = ChatPromptTemplate.from_template(
        """
        아래에 어떤 질문에 대한 웹 검색을 수행한 뒤 정리한 요약 내용입니다

        [검색 요약]
        {mysummary}

        [질문]
        {myquestion}

        위 요약에 있는 정보만 사용해서 질문에 답변하세요
        추측하거나 지어내지 말고 모르면 모른다고 답변하세요

        [최종 답변]

        """
    )
  # Tavily로 실시간 웹 검색
  def summarize_search(self, question):
    raw_result = self.search.invoke( #2) invoke : LLM의 invoke가 아니라 Tavily로 검색해! 라는 invoke
      {"query":question,}
    )
    time.sleep(0.5) # 연속 호출 제한
    chain = self.summary_prompt | self.llm # LangChain 기능 사용

    # LLM에게 요약 생성
    summary_msg = chain.invoke( # 1) invoke : LLM에게 prompt에 대한 답변 요청
        {
            "search_results":raw_result,
        }
    )
    return summary_msg.content # 실제 text값만 return

  def answerFunc(self, question):
    print(f'검색 및 요약 중 : {question}')

    # 1단계 : 질문에 대해  웹 검색 후 요약 생성
    summary = self.summarize_search(question)

    # 2단계 : LLM에게 검색 결과가 요약된 내용을 바탕으로 최종 답변 생성
    chain = self.answer_prompt | self.llm

    answer_msg = chain.invoke(
        {
            "mysummary":summary,
            "myquestion":question,
        }
    )
    return answer_msg.content

In [5]:
if __name__=="__main__":
  web_rag = MyOptimuzerWebRag()
  questions = [
      "최신 의료AI 기술은 뭐니?",
      "한국의 여름철 장마 기간 중 점심 때, 테헤란로 근처에서 먹기 좋은 음식을 추천해",
      "한국의 전망있는 의료AI 회사를 소개해"
  ]

  for q in questions:
    print(f"질문:'{q}'")
    answer = web_rag.answerFunc(q)
    print(f'[답변] \n{answer}\n')

질문:'최신 의료AI 기술은 뭐니?'
검색 및 요약 중 : 최신 의료AI 기술은 뭐니?
[답변] 
최신 의료 AI 기술은 질병 진단과 치료 계획 수립에 큰 영향을 미치고 있습니다. AI는 방대한 의료 데이터를 신속하게 분석하여 조기 진단을 가능하게 하고, 개인 맞춤형 치료 계획을 제안함으로써 치료 효과를 극대화합니다. 또한, 자연어 처리(NLP)와 예측 분석 기술을 통해 의료진과 환자 간의 의사소통을 개선하고, 임상 결정을 지원하는 데 기여합니다. AI의 발전으로 의료 이미지 판독 및 데이터 관리의 효율성이 높아지고 있으며, 이는 환자의 생존율 향상에 기여하고 있습니다.

질문:'한국의 여름철 장마 기간 중 점심 때, 테헤란로 근처에서 먹기 좋은 음식을 추천해'
검색 및 요약 중 : 한국의 여름철 장마 기간 중 점심 때, 테헤란로 근처에서 먹기 좋은 음식을 추천해
[답변] 
여름철 장마 기간 중 테헤란로 근처에서 점심으로 추천할 만한 음식은 다음과 같습니다:

1. **초계탕**: 차가운 닭 육수에 식초와 겨자를 넣어 맛을 낸 전통 음식으로, 여름에 인기가 많습니다.
2. **백합죽**: 은은한 감칠맛과 깔끔한 끝 맛이 특징인 건강식으로, 여름철에 적합합니다.
3. **냉면**: 더운 날씨에 시원하게 즐길 수 있는 간단하면서도 맛있는 선택입니다.
4. **보양식**: 단백질과 미네랄이 풍부한 보양식도 여름철 기운을 북돋아 주는 좋은 선택입니다.

이 외에도 간단한 냉털 레시피와 같은 손쉬운 요리도 여름철에 적합합니다.

질문:'한국의 전망있는 의료AI 회사를 소개해'
검색 및 요약 중 : 한국의 전망있는 의료AI 회사를 소개해
[답변] 
한국의 전망 있는 의료 AI 회사로는 루닛, 셀바스AI, 뷰노, 신테카바이오, 라이프시맨틱스 등이 있습니다. 이들 기업은 AI 기술을 활용하여 의료 진단, 신약 개발, 비대면 진료 플랫폼 등 다양한 분야에서 활동하고 있으며, 최근에는 해외 진출을 위한 MOU 체결과 FDA 인허가를 통해 글로벌 시장에서도 주목받고 있습니다. 한